# So sánh bốn mô hình trên A và A+B

Mở bằng kernel Python của dự án rồi chọn **Run All**. Mỗi mô hình giữ cùng cấu hình và cùng 5 fold phân tầng trên tập phát triển khi so sánh A (36 đặc trưng) với A+B (38 đặc trưng).

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_dir = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "feature_b" / "common.py").is_file()),
    cwd / "ML01_Student_Dropout",
)
script_dir = project_dir / "feature_b"
model_names = {
    "lr": "Logistic Regression",
    "rf": "Random Forest",
    "svm": "SVM",
    "xgb": "XGBoost",
}

for key in model_names:
    script = script_dir / f"run_{key}.py"
    if not script.is_file():
        raise FileNotFoundError(f"Không tìm thấy {script}. Mở dự án ML01_Student_Dropout trong VS Code.")

data_path = project_dir / "data" / "processed" / "development.csv"
if not data_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy {data_path}. Chạy notebooks/01_eda.ipynb trước.")

## Logistic Regression

In [2]:
%run "{script_dir.as_posix()}/run_lr.py"

3539 mẫu | A: 36 cột | A+B: 38 cột
Số mẫu đăng ký 0 môn HK1: 148
model data  features  cv_macro_f1  oof_recall_dropout  oof_recall_enrolled  oof_recall_graduate sklearn_version
   lr    A        36     0.717353            0.733509             0.618898             0.823430           1.9.1
   lr  A+B        38     0.717317            0.730871             0.618898             0.825693           1.9.1
Chênh lệch CV Macro-F1 (A+B trừ A): -0.000037
Đã lưu: C:\Users\Admin\Documents\GitHub\sgu-MAYHOC\ML01_Student_Dropout\experiments\feature_b\lr_a_vs_ab.csv


### CV Macro-F1 gần như giữ nguyên, từ 0,717353 trên A xuống 0,717317 trên A+B. Recall OOF của lớp Dropout giảm nhẹ từ 73,35% xuống 73,09%, còn Enrolled giữ ở mức 61,89%.

## Random Forest

In [3]:
%run "{script_dir.as_posix()}/run_rf.py"

3539 mẫu | A: 36 cột | A+B: 38 cột
Số mẫu đăng ký 0 môn HK1: 148
model data  features  cv_macro_f1  oof_recall_dropout  oof_recall_enrolled  oof_recall_graduate sklearn_version
   rf    A        36     0.716244            0.748461             0.508661             0.877759           1.9.1
   rf  A+B        38     0.723427            0.753738             0.530709             0.874363           1.9.1
Chênh lệch CV Macro-F1 (A+B trừ A): +0.007183
Đã lưu: C:\Users\Admin\Documents\GitHub\sgu-MAYHOC\ML01_Student_Dropout\experiments\feature_b\rf_a_vs_ab.csv


### CV Macro-F1 tăng từ 0,716244 trên A lên 0,723427 trên A+B, chênh lệch +0,007183. Trong lượt chạy này, hai đặc trưng B giúp Random Forest cải thiện điểm kiểm định chéo.

## SVM

In [4]:
%run "{script_dir.as_posix()}/run_svm.py"

3539 mẫu | A: 36 cột | A+B: 38 cột
Số mẫu đăng ký 0 môn HK1: 148
model data  features  cv_macro_f1  oof_recall_dropout  oof_recall_enrolled  oof_recall_graduate sklearn_version
  svm    A        36     0.715364            0.736148             0.595276             0.832484           1.9.1
  svm  A+B        38     0.717298            0.738786             0.595276             0.835314           1.9.1
Chênh lệch CV Macro-F1 (A+B trừ A): +0.001934
Đã lưu: C:\Users\Admin\Documents\GitHub\sgu-MAYHOC\ML01_Student_Dropout\experiments\feature_b\svm_a_vs_ab.csv


### CV Macro-F1 tăng nhẹ từ 0,715364 trên A lên 0,717298 trên A+B, chênh lệch +0,001934.

## XGBoost

In [5]:
%run "{script_dir.as_posix()}/run_xgb.py"

3539 mẫu | A: 36 cột | A+B: 38 cột
Số mẫu đăng ký 0 môn HK1: 148
model data  features  cv_macro_f1  oof_recall_dropout  oof_recall_enrolled  oof_recall_graduate sklearn_version
  xgb    A        36     0.713649            0.760774             0.434646             0.916808           1.9.1
  xgb  A+B        38     0.708953            0.766931             0.423622             0.910017           1.9.1
Chênh lệch CV Macro-F1 (A+B trừ A): -0.004696
Đã lưu: C:\Users\Admin\Documents\GitHub\sgu-MAYHOC\ML01_Student_Dropout\experiments\feature_b\xgb_a_vs_ab.csv


### CV Macro-F1 giảm từ 0,713649 trên A xuống 0,708953 trên A+B, chênh lệch −0,004696 trong lượt chạy này.

## So sánh CV Macro-F1

In [6]:
output_dir = project_dir / "experiments" / "feature_b"
results = pd.concat(
    [pd.read_csv(output_dir / f"{key}_a_vs_ab.csv") for key in model_names],
    ignore_index=True,
)
results["model"] = results["model"].map(model_names)
comparison = results.pivot(index="model", columns="data", values="cv_macro_f1")
comparison = comparison.reindex(index=list(model_names.values()), columns=["A", "A+B"])
comparison["Chênh lệch A+B − A"] = comparison["A+B"] - comparison["A"]
comparison = comparison.rename(columns={"A": "Macro-F1 A", "A+B": "Macro-F1 A+B"})
display(comparison.round(6))

data,Macro-F1 A,Macro-F1 A+B,Chênh lệch A+B − A
model,,,
Logistic Regression,0.717353,0.717317,-0.000037
Random Forest,0.716244,0.723427,0.007183
SVM,0.715364,0.717298,0.001934
XGBoost,0.713649,0.708953,-0.004696


### Nhận xét:
Logistic Regression dẫn đầu trên bộ A với CV Macro-F1 là 0,717353. Khi thêm hai đặc trưng B, Random Forest có mức cải thiện lớn nhất và đạt điểm cao nhất trong các cấu hình đã thử, với 0,723427. Kết quả kiểm định chéo này đưa Random Forest trên A+B vào nhóm ứng viên để đánh giá tiếp trên tập kiểm tra giữ riêng.